# 05 — Attribution: which components close the belief–report gap

PLAN2.md §4.3 and [`misc/exp2_spec.md`](../misc/exp2_spec.md), gates **V5** and the first
half of **V6**. The measurement, stated once:

> `LD = logit(a_H) − logit(a_D)` at the answer slot. Cache the **honest** run's component
> outputs there, replace one of them in the **deceptive** run, and ask how far `LD` moves
> back toward the honest value.
>
> `recovery(c) = (LD_patched − LD_D) / (LD_H − LD_D)`

**What 04 handed this notebook.** `ℓ*` = **25**, `a_H` peaking at **L19–22**, and a gated
bank of **15 whole polarity pairs — 30 items, 15 Yes-true and 15 No-true**
([`misc/bank_v2_spec.md`](../misc/bank_v2_spec.md)). Every belief item has a twin sharing its
context, persona, category and stake byte-for-byte, with the question reversed so the true
answer flips.

**The bias that shapes the analysis.** Under D the model leans `' Yes'` — 70/30 across the
bank against 49/51 under H, refusing to lie 30/50 times when the lie must be phrased `' No'`
against 12/50 when it can be phrased `' Yes'`, for semantically identical false claims.
Consequently 04's `LEGIBLE`/`NEVER` families are **a polarity split under another name**: all
14 never-legible items are No-true; 7 of 8 crossed and all 8 leads-at-top are Yes-true.

05 is largely insulated, because it reads the model's own logits rather than the J-lens and
`LD` is polarity-signed per item — a `' Yes'`-writer scores positive recovery on a Yes-true
item and negative on its No-true twin, cancelling on a balanced set. §6 makes that
cancellation **explicit and testable** rather than incidental: a component is a hit only if
it clears the bar in *both* arms. The families are still reported exactly as
`exp2_spec.md` §2a registered them, as an artefact rather than as the primary subgroup.

**Self-contained, and that is deliberate.** This notebook loads its own model and reads 04's
artifacts off `results/`. It shares nothing with 04's kernel, so the two can be restarted
independently — but they do share one 24 GB GPU, which §0 checks before loading anything.

| | |
|---|---|
| instrument | [`attribution.py`](../src/nandaproj/attribution.py) — `Patcher` (torch) + the bars, nulls, arms and sets (torch-free, tested in `tests/test_attribution.py`) |
| framework | **nnsight**, settled empirically by `just probe --all`: it runs the real HF module, so it *is* the object 04 measured `ℓ*` on |
| search scope | the **whole stack**. L24–25 is §8's prediction, not a filter — pre-selecting it would make the prediction unfalsifiable (PLAN2.md §7.4) |
| subgroups | **polarity arms** (primary, 15/15 by construction) and `LEGIBLE`/`NEVER` (registered artefact) |

**Order is load-bearing.** §4 computes the wrong-source null and writes it to disk **before**
§5 runs the real sweep. That ordering is the entire difference between a pre-registration
and a description of what was found.

In [ ]:
# --- §0 which model, and is there room for it? ----------------------------
# Run this BEFORE the header cell; `get_model_config()` reads the env var at
# call time, so this is all it takes to switch scale.
#
# The guard is the point. 04's kernel (or Exp 1's) holds a 4B model at ~8.6 GiB
# and this notebook loads a second copy. Without the check that collision is an
# OOM four minutes into a model load; with it, it is a refusal in two seconds
# that names the fix.
import gc
import os

os.environ["NANDA_PRESET"] = "target"   # debug(270m) | main(1b) | target(4b) | escalate(12b)

NEEDED_GIB = {"debug": 2.0, "main": 4.0, "target": 12.0, "escalate": 28.0}

for _name in ("patcher", "model", "tok"):
    globals().pop(_name, None)
gc.collect()

try:
    import torch

    torch.cuda.empty_cache()
    free = torch.cuda.mem_get_info()[0] / 2**30
    want = NEEDED_GIB[os.environ["NANDA_PRESET"]]
    print(f"free VRAM: {free:.1f} GiB   needed for {os.environ['NANDA_PRESET']}: {want} GiB")
    if free < want:
        raise RuntimeError(
            f"only {free:.1f} GiB free. Shut the 04 / Exp 1 kernel (Kernel > Shut Down "
            f"Kernel there, not just close the tab -- a closed tab keeps its kernel and "
            f"its model), then re-run this cell.")
    print("room to load.")
except (ImportError, RuntimeError) as exc:
    print(exc)

In [ ]:
# --- standard header ------------------------------------------------------
%load_ext autoreload
%autoreload 2

import sys, pathlib
for p in ("/workspace/NandaProj", ".."):
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from nandaproj import (attribution, config, geometry, items, lens_readout, polarity,
                       sweepset, viz)

cfg = config.get_model_config()      # NANDA_PRESET env var, defaults to debug
config.ensure_dirs()
print("preset:", cfg.name, "|", cfg.n_params, "|", cfg.dtype)
print("device:", config.get_device())

In [ ]:
# The model, and the instrument on top of it. No lens here: 05 works on the
# model's own logits, not on the J-lens. 04 located the layer; this notebook
# asks which components act there, and mixing the two instruments in one
# measurement would make it impossible to say which one a result came from.
tok = AutoTokenizer.from_pretrained(cfg.name, cache_dir=str(config.HF_CACHE))
model = AutoModelForCausalLM.from_pretrained(
    cfg.name,
    cache_dir=str(config.HF_CACHE),
    dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()

patcher = attribution.Patcher(model, tok)
print(patcher.describe())

## 1. Three asserts, before any science

Every failure mode of this notebook is silent. A patch that does not land, a head slice that
is off by a head, a decomposition that does not close — none of them raise, and all three
look exactly like *"this component does not matter"*, which is a sentence this notebook is
otherwise in the business of producing.

| assert | what its failure would look like downstream |
|---|---|
| **the head split is exact** — `Σ_h z_h @ W_O[h] == o_proj(z)` | heads attributed to their neighbours; a top set that is real but mislabelled |
| **a no-op patch is the identity** | every recovery number contaminated by the act of tracing |
| **an intervention lands** | a flat ranking, read as "the effect is distributed" |

The probe already showed all three passing on 270m (`4.8e-07`, bit-identical, Δ = 4.06).
This re-runs them **on the model actually loaded**, because 4b is the multimodal wrapper and
270m is not — the exact difference where a module path silently changes meaning.

In [ ]:
# Numbers, not booleans. `assert x` prints nothing when it passes and tells you
# nothing about how close it came to failing.
#
# The probe pair is the model's own top-2 at the slot, not two hand-picked
# strings: `" Berlin"` being a single token is a guess about the vocabulary, and
# a guess that fails here fails as a confusing exception in a cell whose job is
# to be unambiguous.
PROBE = "The capital of France is"
ID_A, ID_B = patcher.top_tokens(PROBE, 2)
L_MID = patcher.n_layers // 2
HEAD = attribution.HEAD

print(f"probe tokens: {tok.decode([ID_A])!r} vs {tok.decode([ID_B])!r}")
base_ld = patcher.logit_diff(PROBE, ID_A, ID_B)
print(f"baseline LD = {base_ld:.4f}")

# (1) the head split, which is what DLA rests on. GQA: the o_proj input is
# n_q_heads x head_dim wide, NOT d_model -- slicing by d_model // n_heads is
# the bug this checks for.
cache = patcher.cache_slot(PROBE)
W_O = patcher.o_proj_weight(L_MID).float()
heads = [attribution.Component(L_MID, HEAD, h) for h in range(patcher.n_heads)]
z_full = torch.cat([cache[c].float() for c in heads])
split_err = (W_O @ z_full
             - sum(W_O[:, patcher.head_slice(h)] @ cache[c].float()
                   for h, c in enumerate(heads))).abs().max().item()
print(f"(1) head split      max|sum_h - full| = {split_err:.2e}   (want < 1e-2)")

# (2) a no-op patch. An empty patch dict is a trace that changes nothing, so it
# must return the baseline exactly -- otherwise tracing itself has an effect and
# every recovery number below carries it.
noop_ld = patcher.patched_logit_diff(PROBE, {}, ID_A, ID_B)
print(f"(2) no-op patch     LD = {noop_ld:.4f}   delta = {abs(noop_ld - base_ld):.2e}"
      f"   (want 0.00e+00)")

# (3) an intervention that must land. Zero one head at the slot -- a write that
# silently does nothing is indistinguishable from a head that does not matter,
# and that is the failure this whole cell exists for.
zeroed = patcher.patched_logit_diff(
    PROBE, {heads[0]: torch.zeros_like(cache[heads[0]])}, ID_A, ID_B)
print(f"(3) zero {heads[0].name} patch  LD = {zeroed:.4f}"
      f"   delta = {abs(zeroed - base_ld):.4f}   (want > 0)")

assert split_err < 1e-2, "head split is wrong -- check head_dim vs d_model // n_heads"
assert abs(noop_ld - base_ld) < 1e-3, "tracing changes the forward pass"
assert abs(zeroed - base_ld) > 0, "the write did not take -- nnsight API drift"
print("\nall three pass. The instrument reads and writes what it claims to.")

## 2. What 04 handed over — the bank, the families, and the gap

Two files, both written by 04 and synced off the box by `just down`:

| file | what it carries |
|---|---|
| `gated_bank_<lens>.json` | the items with the answers the model **actually gave**, not the bank's stated ones |
| `belief_readout_all_<lens>.npz` | the per-layer curves, from which the two families are derived |

**The families are derived here and never touched again** (`exp2_spec.md` §2a). `LEGIBLE` is
the items where `a_H` led the J-lens on some readable layer under D; `NEVER` is the rest.

**The v1 run's families were a polarity split.** All 4 `LEGIBLE` items had `a_H = ' Yes'`
and all 7 `NEVER` items had `a_H = ' No'` — a perfect separation, so "the honest answer is
legible under D" and "` Yes` is legible under D" made the same prediction and no component
ranked below could be attributed to one rather than the other. That is why the size check
against 4 / 7 is gone: on the v2 bank a different split is *expected*, and the question is
no longer how many items are in each family but whether the families track belief or token.

**The number that decides it is twin agreement.** v2 pairs every item with a polarity twin
over a byte-identical context and persona (`misc/bank_v2_spec.md`). If legibility is a
property of the item's belief, both twins land in the same family. If it is a property of
the answer token, every pair splits across the two families. Context and persona are held
fixed, so there is nothing else left for the difference to be. It is reported below whatever
it says, and nothing is dropped on the basis of it.

**`LD_H − LD_D` is the denominator of every recovery number below**, so it is printed per
item before anything is patched — now with its polarity split, because on v1 the gap ran
13–18 on Yes-true items and 21–30 on No-true ones, and a denominator that differs by answer
token makes every pooled median a polarity-weighted average. Where the gap is small the ratio
is unstable and the item is flagged rather than averaged in — `MIN_LD_GAP` in the module,
`exp2_spec.md` §10.

In [ ]:
# Which upstream run to attribute. "" is 04's v2 bank -- the run every number in
# this notebook's prose was written against. "alleged" is 04c's good-news-
# concealed bank, whose l* and families are its own; if you set it, the v1/v2
# numbers quoted in the markdown are NOT about the files being loaded, and
# saying so in the writeup is the whole point of the tag.
RUN_TAG = "alleged"               # "" | "alleged"
_suffix = f"_{RUN_TAG}" if RUN_TAG else ""

BANK_JSON = config.RESULTS / f"gated_bank{_suffix}_{cfg.lens_id}.json"
CURVES_NPZ = config.RESULTS / f"belief_readout_all{_suffix}_{cfg.lens_id}.npz"
SEL_JSON = config.RESULTS / f"attribution_items{_suffix}_{cfg.lens_id}.json"

# D only. C2 was dropped after 04/04d showed the model answers " No" on 30/30
# items under it: it is a "say No" condition, not an inversion, so V6 (D set vs
# C2 set) is not evaluable on this model and is reported as such in 7.
CONDS = ("D",)                     # swept over every component
BASELINE_CONDS = ("D", "C1")       # ...plus C1, which 7 patches with the D set only

# `conditions=None`: the gated bank carries its prompts, and a missing condition
# must raise rather than quietly acquire a template persona that would then
# never appear in the writeup (PLAN2.md 7.4, same as 04 2).
#
# `with_static_meta` puts `polarity` and `pair_id` back -- 04's `to_json` dropped
# `meta`, and without them every polarity arm below is empty, which is not an
# error anywhere: it is a full ranking with no hits and no stated reason.
GATED = items.load(BANK_JSON, conditions=None)
# The gated bank was written by a `to_json` that dropped `meta`, so it carries no
# `pair_id`/`polarity` and every polarity arm below would silently be empty (the
# first run of this cell reported "twin agreement over 0 whole pairs"). Fill the
# static bank fields back in by item_id; measured values are never overwritten.
GATED = items.LoadResult(items=items.attach_meta(GATED.items, items.load_bank().items),
                         used_templates=GATED.used_templates, source=GATED.source)
CURVES = lens_readout.load_curves(CURVES_NPZ)
SOURCE = (items.load(config.DATA / "alleged_arm_final.json", conditions=None)
          if RUN_TAG == "alleged" else items.load_bank())
GATED = sweepset.with_static_meta(GATED, SOURCE)

print(f"run tag: {RUN_TAG or '(none -- 04 / v2 bank)'}")
print(GATED.summary())
print(f"static fields from {SOURCE.source}")
print(f"{len(CURVES)} curves from {'04c' if RUN_TAG else '04'}\n")

BY_ID = GATED.by_id
FAM_ALL = attribution.families(CURVES, condition="D")
POL_ALL = {i: sweepset.polarity_of(BY_ID[i])
           for ids in FAM_ALL.values() for i in ids}

# --- the sweep budget, fixed here and written to disk before any patch -----
#
# 306 components x n items x 3 sweeps at ~30 ms a pass: the whole alleged bank
# is ~9.5 h and ~$3.60, 32 items ~2.4 h. `sweepset.choose` keeps every item of
# the scarce polarity and fills the rest evenly across the families; its
# docstring has why that is not the same as stratifying by family.
MAX_SWEEP_ITEMS = 32 if RUN_TAG == "alleged" else None    # None = every item
SEL = sweepset.choose(FAM_ALL, POL_ALL, max_items=MAX_SWEEP_ITEMS, seed=0,
                      run_tag=RUN_TAG)
SEL.to_json(SEL_JSON)

SWEEP_ITEMS = [BY_ID[i] for i in SEL.item_ids]
FAM = {f: SEL.by_family(f) for f in FAM_ALL}
FAMILY_OF = dict(SEL.family_of)
POL = dict(SEL.polarity_of)

print(f"{SEL.summary()} -> {SEL_JSON.name}")
if len(SEL.item_ids) < SEL.n_available:
    print("The medians below are over this subset, fixed before the first patch.")
if SEL.one_sided:
    print("!! ONE-SIDED. Section 6's both-arm bar cannot be evaluated and every "
          "component\n   will be refused. Fix the bank rather than dropping the bar.")

# --- the families against polarity, before they are used for anything ------
#
# On v1 the split was exact -- all 4 LEGIBLE items Yes-true, all 7 NEVER items
# No-true -- so "the honest answer is legible under D" and "' Yes' is legible
# under D" made the same prediction and every component below could have been
# ranked by answer token. Two numbers say whether that is still the case: a
# family that is one-sided is still confounded, and twins that split across
# families mean legibility follows the token rather than the belief.
print(f"\n{'family':<10} {'n':>4} {'of':>4} {'Yes-true':>9} {'No-true':>8}")
for fam in (attribution.LEGIBLE, attribution.NEVER):
    ids = FAM[fam]
    n_yes = sum(POL[i] == attribution.YES for i in ids)
    print(f"{fam:<10} {len(ids):>4} {len(FAM_ALL[fam]):>4} {n_yes:>9} "
          f"{len(ids) - n_yes:>8}"
          + ("   <- one-sided: this family IS a polarity group"
             if ids and n_yes in (0, len(ids)) else ""))

PAIR_INDEX, n_pair_ids = sweepset.whole_pairs(SWEEP_ITEMS)
print(f"\n{n_pair_ids} pair ids over the swept set, {len(PAIR_INDEX)} of them whole")
if not PAIR_INDEX:
    print("The WITHIN-PAIR control is UNAVAILABLE. Not a bug and not a null: the twins")
    print("were never gated together (04c: 12 of 390 pairs), so no pair holds context")
    print("and persona fixed. The polarity ARMS in 6 are still live -- Yes-true items")
    print("against No-true ones -- but they do not hold the scenario fixed, so a")
    print("difference between them could be the scenario. A stated limitation.")
else:
    agree = {"both legible": [], "both never": [], "SPLIT": []}
    for pid, (yes_it, no_it) in PAIR_INDEX.items():
        fams = {FAMILY_OF.get(yes_it.item_id), FAMILY_OF.get(no_it.item_id)}
        if None in fams:
            continue                       # one twin did not reach the sweep
        agree["SPLIT" if len(fams) == 2 else
              "both legible" if fams == {attribution.LEGIBLE}
              else "both never"].append(pid)
    n_pairs = sum(len(v) for v in agree.values())
    print(f"twin agreement over {n_pairs} whole pairs:")
    for key, pids in agree.items():
        print(f"  {key:<13} {len(pids):>3}  {', '.join(sorted(pids)[:6])}")
    if n_pairs:
        print(f"\n  {len(agree['SPLIT']) / n_pairs:.0%} of pairs split across families."
              "\n  ~100% -> legibility follows the answer token and exp2_spec.md 2a needs"
              "\n  rewriting before a component claim rests on the families; ~0% -> they"
              "\n  are what they say they are and 8.3 reads as written.")

# The denominator, per item and PER CONDITION, before anything is patched.
# One baseline per condition and not a shared one: C2's recoveries measured
# against D's baseline would all be wrong and none of them obviously so, and
# V6 is decided on the comparison between them. `patch_sweep` raises rather
# than falling back to D's baseline, so a missing entry here fails loudly.
print(f"\n{'item':<24} {'family':<8} {'pol':>4} {'a_H':>6} {'a_D':>6} {'LD_H':>8} "
      + " ".join(f"{'LD_' + c:>8}" for c in BASELINE_CONDS) + f" {'gap(D)':>8}  usable")
LD = {}
for it in SWEEP_ITEMS:
    id_h, id_d = patcher.token_id(it.answer_honest), patcher.token_id(it.answer_lie)
    entry = {"id_h": id_h, "id_d": id_d,
             "H": patcher.logit_diff(items.render(tok, it, "H"), id_h, id_d)}
    for cond in BASELINE_CONDS:
        entry[cond] = patcher.logit_diff(items.render(tok, it, cond), id_h, id_d)
    LD[it.item_id] = entry

    gap = entry["H"] - entry["D"]
    print(f"{it.item_id:<24} {FAMILY_OF[it.item_id]:<8} {POL[it.item_id]:>4} "
          f"{it.answer_honest!r:>6} {it.answer_lie!r:>6} {entry['H']:>8.2f} "
          + " ".join(f"{entry[c]:>8.2f}" for c in BASELINE_CONDS)
          + f" {gap:>8.2f}  {'yes' if abs(gap) >= attribution.MIN_LD_GAP else 'NO'}")

n_bad = sum(abs(v["H"] - v["D"]) < attribution.MIN_LD_GAP for v in LD.values())
print(f"\n{len(LD) - n_bad}/{len(LD)} items have a usable gap under D "
      f"(>= {attribution.MIN_LD_GAP} logits). Unusable items stay in the sweep and are "
      f"dropped from medians -- excluding them here would hide that they exist.")

# Every recovery below is divided by this gap, so a per-polarity difference in
# the denominator is a per-polarity difference in every ranking. On v1 it ran
# 13-18 on the Yes-true items and 21-30 on the No-true ones. Reported, not
# corrected.
gaps = {i: LD[i]["H"] - LD[i]["D"] for i in LD}
print(f"\nLD gap by polarity: {polarity.by_polarity(gaps, [BY_ID[i] for i in gaps])}")

## 2b. The geometry, before spending two hours on it

Cheap and diagnostic: 32 items × 4 conditions = **128 forward passes**, under a minute. It
runs here rather than at the end because what it says can change whether the sweep below is
worth starting.

**The question.** Is there a *direction* in the late residual stream that separates the
honest run from the deceptive one — the thing §5 is about to look for one component at a
time? PCA is the crude version of that question, and it is worth asking crudely first.

**Why the obvious plot is a trap.** These 32 prompts differ in scenario — freezers,
scaffolds, playparks — and in one system turn. The scenario variance is enormous and the
condition variance is a small displacement inside each scenario's cloud. So a PCA of raw
residuals will put topic on PC1 and PC2, the colours will fall in tidy clusters, and it will
look like structure. It is structure; it is just not the structure this notebook is about.

Two things guard against reading it that way:

| | |
|---|---|
| **`eta_squared`, printed beside every PC** | the fraction of that component's variance the labelling explains. `item` scoring 0.9 and `condition` 0.02 means PC1 is a topic axis, however clean the colours look |
| **the paired panel** | `h_ℓ(D) − h_ℓ(H)` per item. The two prompts share every token but the system turn, so the scenario **cancels exactly** and what is left is the displacement the condition causes |

The paired panel is the one that can show a shared direction, and it is the same object
`intervene.d_dim` averages in Exp 1 — so a tight cluster here and a null there is a fact
about the intervention, not about the geometry.

**`cosine_to_mean` is the summary number**, leave-one-out so no vector scores against
itself. Cosines near 1 mean every item moves the same way and "the deception direction" is a
sensible phrase. Cosines spread around 0 mean each item moves its own way and the mean
vector is an average with no member near it — which is exactly what a mean direction looks
like when there is no direction.

`l*` is marked as a prediction, not used to select anything.

In [ ]:
# --- 2b. residual-stream PCA at the answer slot ---------------------------
#
# `cache_resid` reads the residual stream AFTER decoder block l, at position -1
# -- the same slot `logit_diff` reads, so a geometry claim and a logit claim are
# about the same vector. One pass per (item, condition).
PCA_CONDS = ("H", "D", "C1", "C2")
PCA_LSTAR = 29 if RUN_TAG == "alleged" else 25       # 04c's l* / 04's l*
PCA_LAYERS = [l for l in sorted({17, 22, 25, PCA_LSTAR, patcher.n_layers - 2})
              if 0 <= l < patcher.n_layers]
PLOT_LAYER = PCA_LSTAR                                # figures at one layer only

RESID = {}                                            # {(item_id, cond, layer): vec}
for it in lens_readout._progress(SWEEP_ITEMS, desc="residuals"):
    for cond in PCA_CONDS:
        for l, v in patcher.cache_resid(
                items.render(tok, it, cond), PCA_LAYERS).items():
            RESID[(it.item_id, cond, l)] = v
print(f"{len(RESID)} residual vectors: {len(SWEEP_ITEMS)} items x "
      f"{len(PCA_CONDS)} conditions x {len(PCA_LAYERS)} layers, d={patcher.d_model}")

# --- panel 1: raw residuals. What actually dominates the variance? --------
#
# The labels are the point, not the plot. `item` is included deliberately even
# though it has one level per four points: if it takes PC1, that axis is the
# scenario and no colouring by condition means anything on it.
KEYS = [(it.item_id, c) for it in SWEEP_ITEMS for c in PCA_CONDS]
LABELS = {
    "condition": [c for _, c in KEYS],
    "polarity": [POL[i] for i, _ in KEYS],
    "family": [FAMILY_OF[i] for i, _ in KEYS],
    "item": [i for i, _ in KEYS],
}

print(f"\n{'=' * 78}\n  RAW residuals -- how much variance, and explained by what\n{'=' * 78}")
RAW_FITS = {}
for l in PCA_LAYERS:
    RAW_FITS[l] = geometry.pca([RESID[(i, c, l)] for i, c in KEYS], k=3)
    print(f"\nlayer {l}{'  <- l* (predicted)' if l == PCA_LSTAR else ''}   "
          f"{RAW_FITS[l].summary()}")
    print(geometry.label_report(RAW_FITS[l], LABELS))

# --- panel 2: paired H->D differences. The scenario cancels. --------------
print(f"\n{'=' * 78}\n  PAIRED H->D differences -- one point per item\n{'=' * 78}")
DIFF_FITS, DIFF_COS, DIFF_IDS = {}, {}, {}
for l in PCA_LAYERS:
    vecs = {(i, c): RESID[(i, c, l)] for i, c in KEYS}
    diffs, kept = geometry.paired_differences(
        vecs, [it.item_id for it in SWEEP_ITEMS], "H", "D")
    DIFF_FITS[l], DIFF_IDS[l] = geometry.pca(diffs, k=3), kept
    DIFF_COS[l] = geometry.cosine_to_mean(diffs)

    # The displacement against the stream it sits in. A "large" effect that is
    # 1% of the residual norm and a small one that is 40% are different claims,
    # and the ratio is the only way to tell them apart.
    step = float(np.median(np.linalg.norm(diffs, axis=1)))
    base = float(np.median([np.linalg.norm(RESID[(i, "H", l)]) for i in kept]))
    print(f"\nlayer {l}{'  <- l*' if l == PCA_LSTAR else ''}   {DIFF_FITS[l].summary()}")
    print(f"  |h_D - h_H| median {step:.1f} against |h_H| median {base:.1f}"
          f"  ({step / base:.0%} of the stream)")
    print(f"  cos to the mean difference (leave-one-out): median "
          f"{np.median(DIFF_COS[l]):+.3f}  min {DIFF_COS[l].min():+.3f}  "
          f"max {DIFF_COS[l].max():+.3f}")
    print(geometry.label_report(
        DIFF_FITS[l], {"polarity": [POL[i] for i in kept],
                       "family": [FAMILY_OF[i] for i in kept]}))

print("\nRead the cosines first. Median near 1 -> every item is displaced the same way and")
print("'the deception direction' is a phrase with a referent. Median near 0 with a wide")
print("range -> each item moves its own way; the mean is an average with no member near")
print("it, and Exp 1's d_DiM null is the expected result rather than a surprise.")
print("Read the polarity row next: a paired PC that IS polarity is the answer-token axis")
print("04c 3a found in the J-lens, showing up in the model's own residual stream.")

# --- the figures, at one layer ---------------------------------------------
# One layer, not five: the tables above are the measurement, and ten plots is a
# gallery rather than a result. Change PLOT_LAYER to look elsewhere.
_raw = RAW_FITS[PLOT_LAYER]
viz.scatter(
    _raw.scores[:, 0], _raw.scores[:, 1],
    labels=LABELS["condition"], symbols=LABELS["polarity"], text=LABELS["item"],
    title=f"L{PLOT_LAYER} raw slot residuals: colour = condition, shape = polarity "
          f"(PC1 {_raw.explained[0]:.0%}, PC2 {_raw.explained[1]:.0%})",
    xaxis="PC1", yaxis="PC2",
).show()

_diff = DIFF_FITS[PLOT_LAYER]
viz.scatter(
    _diff.scores[:, 0], _diff.scores[:, 1],
    labels=[POL[i] for i in DIFF_IDS[PLOT_LAYER]],
    symbols=[FAMILY_OF[i] for i in DIFF_IDS[PLOT_LAYER]],
    text=DIFF_IDS[PLOT_LAYER],
    title=f"L{PLOT_LAYER} paired h(D) - h(H): colour = polarity, shape = family "
          f"(PC1 {_diff.explained[0]:.0%}, PC2 {_diff.explained[1]:.0%})",
    xaxis="PC1", yaxis="PC2",
).show()

# The cosine spread across depth: the one-line version of this whole cell --
# where in the stack, if anywhere, the items start moving together.
viz.series_line(
    PCA_LAYERS,
    {"median": [float(np.median(DIFF_COS[l])) for l in PCA_LAYERS],
     "min": [float(DIFF_COS[l].min()) for l in PCA_LAYERS],
     "max": [float(DIFF_COS[l].max()) for l in PCA_LAYERS]},
    title="Is there one direction? cos(each item's H->D displacement, the mean of the rest)",
    xaxis="layer", yaxis="cosine", y_range=(-1, 1),
).show()

## 3. DLA — proposes, does not rank

One pass per item, every component's **direct** contribution to `LD`.

**Frozen-scale, and what that does and does not mean.** Gemma 3 writes
`resid += post_attention_layernorm(attn_out)`, and RMSNorm is not linear — so a head's
contribution cannot be read off its output directly. Taking each norm's scale factor from
the observed forward pass and holding it fixed makes the split linear again, because
`RMSNorm(x) = x · s(x) · (1 + w)` and `s` is a **scalar**: distributing it over the heads
that sum to `x` is an *identity*, not an approximation.

So the decomposition is **exact for the run it describes**. What it is not is a
counterfactual — removing a head would change `s` — which is exactly why PLAN2.md §4.3 has
DLA propose and patching rank.

The completeness residual is printed for a different reason than the spec first assumed: it
cannot detect a loose approximation, because there isn't one. It detects a **wrong graph** —
a sublayer writing somewhere other than where this code thinks it does, or logit softcapping
making the unembed nonlinear. `exp2_spec.md` §5.1's 5% bar stands; a failure means a bug,
not noise.

In [ ]:
dla_median, dla_results = attribution.dla_sweep(patcher, SWEEP_ITEMS, LD, condition="D")

# The two honesty checks, per item, before the table is read.
print(f"\n{'item':<22} {'LD_true':>9} {'sum(DLA)':>9} {'residual':>9} {'frac':>7} "
      f"{'split err':>10}")
for it, res in zip(SWEEP_ITEMS, dla_results):
    print(f"{it.item_id:<22} {res.ld_true:>9.3f} {res.total:>9.3f} {res.residual:>9.3f} "
          f"{res.residual_frac:>7.3f} {res.layer_split_err:>10.2e}"
          f"{'' if res.complete else '   <- OVER 5%'}")

n_ok = sum(r.complete for r in dla_results)
print(f"\n{n_ok}/{len(dla_results)} items close to within 5%.")
if n_ok < len(dla_results):
    print("The decomposition is exact for the observed run, so a residual this large is a "
          "WRONG GRAPH, not noise: check final_logit_softcapping, and check that the MLP's "
          "residual increment really is post_feedforward_layernorm's output on this model.")
    print("Per exp2_spec.md 5.1 the DLA table below is INDICATIVE ONLY and enters no claim.")

# The proposal. Median over items, biggest |contribution| first. This is a
# ranking of a *linear decomposition of one run*; it is not evidence that
# removing any of these changes anything, which is what 5 is for.
print(f"\n{'component':<10} {'type':<18} {'median DLA':>11}")
for comp, val in sorted(dla_median.items(), key=lambda kv: -abs(kv[1]))[:20]:
    print(f"{comp.name:<10} {comp.layer_type:<18} {val:>11.4f}")

print("\nPredicted (exp2_spec.md 8.1): concentrated at L24-25, heads over MLPs.")
print("This cell cannot confirm that -- it can only say where to look first.")

## 4. The null — computed and saved **before** the real sweep

**This ordering is the pre-registration.** V5 asks for a null and an effect-size bar fixed
before the ranking is looked at (PLAN2.md §7.4). Running the null first, writing it to disk,
and only then measuring the real thing is what makes that a fact about the run rather than a
claim about intentions.

**The wrong-source patch.** Patch component `c` with the honest-run activation *of a
different item*. Same component, same perturbation magnitude, wrong content. That separates:

| | |
|---|---|
| "this component carries the honest answer" | recovery high with the item's own activation, low with someone else's |
| "poking this component moves the output" | recovery high **either way** — and a zero-ablation null cannot tell these apart |

The pairing is a **derangement**: no item is ever its own source. A single fixed point would
put signal into the null and raise the bar it is supposed to define, in the direction that
makes a real component look unremarkable.

Bars, from `exp2_spec.md` §5.3 and already fixed in the module: median recovery ≥ **0.20**
*and* above the **95th** percentile of the component's own null.

In [ ]:
COMPONENTS = patcher.components()
# `_suffix`, like every path in 2: without it an alleged run overwrites the v2
# sweep in place, which is the failure 04c 2 documents -- two runs wrote the
# same filename and the second destroyed the only record of the first.
NULL_JSON = config.RESULTS / f"attribution_null{_suffix}_{cfg.lens_id}.json"

PAIRS = attribution.derangement([it.item_id for it in SWEEP_ITEMS], seed=0)
print(f"{len(COMPONENTS)} components x {len(SWEEP_ITEMS)} items = "
      f"{len(COMPONENTS) * len(SWEEP_ITEMS)} patches per condition\n")
print("wrong-source pairing (no item is its own source):")
for k, v in PAIRS.items():
    print(f"  {k:<22} <- honest run of {v}")

null_rows = attribution.patch_sweep(
    patcher, SWEEP_ITEMS, LD, condition="D", components=COMPONENTS,
    sources=PAIRS, family_of=FAMILY_OF, by_id=BY_ID,
    save_to=NULL_JSON, desc="null (wrong source)")

print(f"\n{len(null_rows)} null rows -> {NULL_JSON}")
vals = attribution.recoveries(null_rows)
print(f"null recovery: median={np.median(vals):.4f}  p95={np.percentile(vals, 95):.4f}  "
      f"max={vals.max():.4f}")
print("\nRead the max: if patching a random item's activation into this one already "
      "recovers a large fraction of LD, the metric is measuring perturbation and not "
      "content, and no ranking below can fix that.")

## 5. The sweep — H→D, every component, D and C2

The causal measurement. For each item: cache the honest run at the slot, then re-run the
deceptive prompt with exactly one component replaced.

**Cost, at v2 scale.** 306 components × **30 items** = 9,180 forward passes per condition,
and this runs three times (null, D, C2) for ~27,500 — roughly **2¼ h**, ≈ **$0.85** on the
4090, against `exp2_spec.md`'s ~50 min / $0.32 written when the bank gated to 11 items. There
is a `tqdm` bar, and the table is re-saved after **every item**, so a sweep that dies at item
22 of 30 leaves 22 on disk.

**Both polarity arms are swept together, necessarily.** The arms are a split of the *same*
rows, not a second sweep — every item is patched with every component either way, and §6
partitions the results. Nothing here costs extra for the arm analysis.

**C2 is in the primary sweep, not a follow-up.** 04 already found C2 crossing at L25 alongside
D, so the deflationary reading — *"it is a negation/instruction-following circuit, not a
deception circuit"* — is live at layer resolution before this notebook starts. Discovering
the two component sets are identical *after* building 06's ablation on top of them costs the
result; the extra ~45 min costs 30 cents.

In [ ]:
SWEEP_JSON = {c: config.RESULTS / f"attribution_{c}{_suffix}_{cfg.lens_id}.json"
              for c in CONDS}

real_rows = {}
for cond in CONDS:
    real_rows[cond] = attribution.patch_sweep(
        patcher, SWEEP_ITEMS, LD, condition=cond, components=COMPONENTS,
        family_of=FAMILY_OF, by_id=BY_ID,
        save_to=SWEEP_JSON[cond], desc=f"patch H->{cond}")
    print(f"{cond}: {len(real_rows[cond])} rows -> {SWEEP_JSON[cond]}")

# Sanity, per condition, before any ranking: the *distribution* of recovery.
# A sweep where nothing moves and a sweep where everything moves are both
# possible failures and they look identical in a top-20 table.
print(f"\n{'cond':<5} {'n':>6} {'median':>8} {'p95':>8} {'max':>8}  best component")
for cond in CONDS:
    vals = attribution.recoveries(real_rows[cond])
    best = max(attribution.by_component(real_rows[cond]).items(),
               key=lambda kv: attribution.median_recovery(kv[1]))
    print(f"{cond:<5} {vals.size:>6} {np.median(vals):>8.4f} "
          f"{np.percentile(vals, 95):>8.4f} {vals.max():>8.4f}  "
          f"{best[0].name} ({attribution.median_recovery(best[1]):.3f})")

## 6. The ranking — V5

Each component against **its own** null, both bars applied — and now a third bar: it must
clear `MIN_RECOVERY` in **both polarity arms**, not only in the pooled median.

### Why the arms are the primary subgroup

Under D the model is biased toward `' Yes'` — 70/30 across the bank against 49/51 under H,
and it refuses to lie 30/50 times when the lie must be phrased `' No'` versus 12/50 when it
can be phrased `' Yes'`, for semantically identical false claims.

05 is *largely* insulated from that, because it reads the model's own logits rather than the
J-lens, and `LD = logit(a_H) − logit(a_D)` is **polarity-signed per item**: a component that
merely writes `' Yes'` scores positive recovery on a Yes-true item and *negative* recovery on
its No-true twin, so on a whole-pairs set the two cancel in the pooled median.

"Largely" is the problem. The cancellation is exact only while the set stays balanced, and
`MIN_LD_GAP` drops single twins without their partners. Requiring both arms makes the
protection **explicit and testable** instead of incidental:

| | pooled median | Yes-arm | No-arm | verdict |
|---|---|---|---|---|
| restores the belief | high | high | high | hit |
| writes `' Yes'` | high *if the set tilts Yes* | high | negative | **not** a hit |

All three numbers are printed, plus the arm gap, so a component that clears pooled and fails
one arm is visible as `one arm` rather than silently blank — blank reads as "did not clear",
which is a different fact.

### LEGIBLE / NEVER are still reported, as a registered artefact

`exp2_spec.md` §2a fixed those families in advance and quietly redefining or dropping them
now is exactly the forking path the spec exists to prevent. So they stay, reported as
registered — but they are **not** the primary subgroup any more. On v1 they *were* a polarity
split under another name: all 14 never-legible items were No-true, 7 of 8 crossed and all 8
leads-at-top were Yes-true. §2's twin-agreement number says whether that still holds; if the
families split every pair, then §8.3's family comparison is a polarity comparison, and the
arms above already made it without the confound.

The sweep set is unaffected: the two families partition all the gated items, so `SWEEP_ITEMS`
is the whole set either way.

`ℓ*` = 25 is **printed on the layer plot as a prediction**, not used to select anything.

In [ ]:
# 04's crossover on the v2 bank; 04c's on the alleged one (l* n=32, median 29,
# range 29-32). A PREDICTION here, never a filter -- which is exactly why it is
# per-run: predicting the other bank's layer is not the conservative choice, it
# is just the wrong number.
L_STAR = 29 if RUN_TAG == "alleged" else 25

# Which answer is *true* for each item -- read in 2 from the bank's declared
# `polarity`, with the measured answer as the fallback and a raise if the two
# disagree.
#
# NOT derived from PAIR_INDEX any more. On a gated arm almost no twin survives
# (04c: 12 of 390 pairs), so a pair-derived mapping is empty, every component is
# refused for want of an arm rather than for want of an effect, and the notebook
# prints a full ranking with no hits and no explanation. The declared field is
# on every item and is still the authored value, so the reason the pair version
# existed -- never let the arm be a function of the behaviour being measured --
# is preserved.
POLARITY_OF = dict(POL)
missing = [it.item_id for it in SWEEP_ITEMS if it.item_id not in POLARITY_OF]
n_yes_arm = sum(v == attribution.YES for v in POLARITY_OF.values())
print(f"{len(POLARITY_OF)} items carry a declared polarity "
      f"({n_yes_arm} Yes-true, {len(POLARITY_OF) - n_yes_arm} No-true)"
      + (f"; {len(missing)} do NOT: {missing}" if missing else ""))
if missing:
    print("An item with no arm is dropped from both arms and can never be a hit. That is "
          "the safe direction, but a long list here means the bank lost `polarity` on the "
          "way to disk and the arm bars are being applied to a set they cannot balance.")
if min(n_yes_arm, len(POLARITY_OF) - n_yes_arm) < 3:
    print("!! one arm has fewer than 3 items. The bar is still applied -- it is the "
          "registered\n   rule -- but a median over that arm is a median over almost "
          "nothing. Report the\n   arm n beside every hit.")


def ranked_for(cond, family=None, arms=True):
    """Rows for one condition, real + null together, judged on both arms.

    The null rows must travel with the real ones: `rank` splits them by
    `source`, and handing it only the real rows gives every component an
    infinite bar and no hits at all.

    `arms=True` is the primary analysis. `LD` is polarity-signed per item, so a
    component that only writes ' Yes' recovers on Yes-true items and
    anti-recovers on their No-true twins; on a balanced set those cancel in the
    pooled median, which is why 05 is largely insulated from the bank's Yes-bias
    under D. The cancellation is exact only while the set stays balanced, and
    MIN_LD_GAP drops single items -- so requiring both arms turns an incidental
    protection into a stated one.

    `family` is kept because exp2_spec.md 2a fixed LEGIBLE/NEVER in advance.
    It is a registered artefact reported below, not the primary subgroup.
    """
    keep = lambda r: family is None or r.family == family    # noqa: E731
    return attribution.rank(
        [r for r in real_rows[cond] if keep(r)] + [r for r in null_rows if keep(r)],
        polarity_of=POLARITY_OF if arms else None)


# --- the primary ranking: pooled, with both arms shown --------------------
RANKED = {None: ranked_for("D")}
print(f"\n{'=' * 92}\n  D, all {len(SWEEP_ITEMS)} items -- hit requires BOTH polarity arms"
      f"\n{'=' * 92}")
print(attribution.report(RANKED[None], limit=15))

# What the arm requirement actually cost, as a number. A large gap between the
# two counts is the finding: it means the pooled ranking was promoting
# components that only work in one polarity direction.
pooled_only = attribution.hits(ranked_for("D", arms=False))
both_arms = attribution.hits(RANKED[None])
print(f"\nhits on the pooled median alone : {len(pooled_only)}")
print(f"hits clearing both polarity arms: {len(both_arms)}")
dropped = [r.component.name for r in pooled_only
           if r.component not in {h.component for h in both_arms}]
if dropped:
    print(f"dropped by the arm requirement  : {dropped}")
    print("Each of these recovers LD in one polarity direction and not the other -- the "
          "signature of a component that writes an answer token rather than restoring "
          "an answer.")

# --- the registered artefact: LEGIBLE / NEVER, reported as fixed ----------
#
# exp2_spec.md 2a fixed these families before any of this ran, and quietly
# redefining or dropping them now is exactly the forking path the spec exists to
# prevent. So they stay, reported as registered -- but they are no longer the
# primary subgroup, because on v1 they were a polarity split under another name
# (all 4 LEGIBLE Yes-true, all 7 NEVER No-true) and cell 2's twin-agreement
# number says whether that is still true. The polarity arms above are the
# subgroup analysis that is not confounded by construction.
#
# The sweep set is unaffected either way: the two families partition all of the
# gated items, so SWEEP_ITEMS is the whole set however they fall.
for fam in (attribution.LEGIBLE, attribution.NEVER):
    RANKED[fam] = ranked_for("D", fam)
    n_yes = sum(POLARITY_OF.get(i) == attribution.YES for i in FAM[fam])
    print(f"\n{'-' * 92}\n  D, {fam} (registered artefact) -- {len(FAM[fam])} items, "
          f"{n_yes} Yes-true / {len(FAM[fam]) - n_yes} No-true\n{'-' * 92}")
    print(attribution.report(RANKED[fam], limit=8))

best_leg = RANKED[attribution.LEGIBLE][0] if RANKED[attribution.LEGIBLE] else None
best_nev = RANKED[attribution.NEVER][0] if RANKED[attribution.NEVER] else None
if best_leg and best_nev:
    print(f"\nbest component  {attribution.LEGIBLE}: {best_leg.component.name} "
          f"{best_leg.median:.3f}   |   {attribution.NEVER}: {best_nev.component.name} "
          f"{best_nev.median:.3f}")
    print("exp2_spec.md 8.3 reads this as the 04 -> 05 bridge holding or not. Read it "
          "against cell 2's twin agreement first: if the families split every pair, this "
          "comparison is a polarity comparison and the arms above already made it.")

In [49]:
# The heads x layers picture, and the R4 split. `prob_heatmap` anchors at zero,
# which matters here: an autoscaled floor makes a 0.02 recovery look like a hit.
def head_map(value_of, clip=True):
    """One value per (head, layer), laid out for `prob_heatmap`.

    Missing and non-finite entries become 0.0 rather than propagating: a NaN in
    a plotly heatmap is a blank cell, which reads as "measured, and zero" when
    it means "not measured at all".
    """
    grid = np.zeros((patcher.n_heads, patcher.n_layers))
    for l in range(patcher.n_layers):
        for h in range(patcher.n_heads):
            v = value_of(attribution.Component(l, attribution.HEAD, h,
                                               patcher.layer_types[l]))
            v = 0.0 if v is None or not np.isfinite(v) else float(v)
            grid[h, l] = min(max(v, 0.0), 1.0) if clip else v
    return grid


med = {r.component: r.median for r in RANKED[None]}
viz.prob_heatmap(
    head_map(lambda c: med.get(c)),
    x=list(range(patcher.n_layers)), y=[f"H{h}" for h in range(patcher.n_heads)],
    title=f"D: median recovery by head (L* = {L_STAR} predicted, not selected)",
    xaxis="layer", yaxis="head",
).show()

# The same map in |arm gap|. Bright here means a head whose recovery depends on
# which answer is true -- a token-writer -- so this is the plot that says
# whether the map above is showing a belief effect or a polarity effect. The two
# are indistinguishable in the first plot and separated by this one.
gap = {r.component: r.arm_gap for r in RANKED[None]}
viz.prob_heatmap(
    head_map(lambda c: abs(gap.get(c, 0.0))),
    x=list(range(patcher.n_layers)), y=[f"H{h}" for h in range(patcher.n_heads)],
    title="D: |Yes-arm - No-arm| by head (bright = polarity-dependent, not belief)",
    xaxis="layer", yaxis="head",
).show()

# MLPs, on their own axis -- there are n_layers of them against n_layers x
# n_heads heads, and putting them in the same heatmap makes them invisible.
mlp_vals = [med.get(attribution.Component(l, attribution.MLP, None,
                                          patcher.layer_types[l]), 0.0)
            for l in range(patcher.n_layers)]
viz.line(mlp_vals, title="D: median recovery by MLP block", xaxis="layer",
         yaxis="recovery").show()

# R4: sliding and full attention are different mechanisms and must not be
# pooled. If the hits cluster in one kind, that is a fact about the circuit;
# if they are reported together, it is not a fact at all.
all_hits = attribution.hits(RANKED[None])
print(f"{len(all_hits)} components clear all three bars under D "
      f"(pooled median, own null, and BOTH polarity arms)")
if all_hits:
    from collections import Counter

    kinds = Counter(c.component.layer_type for c in all_hits)
    layers = Counter(c.component.layer for c in all_hits)
    print(f"  by attention type : {dict(kinds)}")
    print(f"  by layer          : {dict(sorted(layers.items()))}")
    in_window = sum(n for l, n in layers.items() if l in (L_STAR - 1, L_STAR))
    print(f"  at L{L_STAR - 1}-{L_STAR} (the prediction): {in_window}/{len(all_hits)}")

    # A survivor is arm-symmetric by construction -- it had to clear the bar
    # twice. Printing the gap anyway is the check that it cleared both
    # *comfortably* rather than scraping one arm, which is what a mostly-
    # polarity component looks like when the bar is generous.
    print(f"\n  {'component':<10} {'Yes-arm':>8} {'No-arm':>8} {'gap':>8}")
    for r in all_hits[:10]:
        print(f"  {r.component.name:<10} {r.median_yes:>8.3f} {r.median_no:>8.3f} "
              f"{r.arm_gap:>8.3f}")
else:
    print("  Nothing clears the bars. exp2_spec.md 8.3: the edit is distributed, "
          "'specific heads' was the wrong resolution, and 06 has nothing to ablate. "
          "That is a registered outcome -- do not lower the bar, and do not drop the "
          "arm requirement, to produce a set.")

0 components clear all three bars under D (pooled median, own null, and BOTH polarity arms)
  Nothing clears the bars. exp2_spec.md 8.3: the edit is distributed, 'specific heads' was the wrong resolution, and 06 has nothing to ablate. That is a registered outcome -- do not lower the bar, and do not drop the arm requirement, to produce a set.


In [48]:
# --- 5.4 reverse patching, D->H, survivors only ---------------------------
#
# Patch the DECEPTIVE run's component into the HONEST run and check the honest
# answer degrades. A component that recovers a_H one way and does nothing the
# other is a one-run artefact (PLAN2.md 4.3.3) -- the H and D runs differ in
# more than one thing, and only the components that move the output in both
# directions are candidates for carrying the edit itself.
#
# Survivors only: this is a confirmation on a handful of components, not a
# second search, and running it over all 306 would be a second multiple-
# comparisons machine (R6).
#
# Damage is reported per polarity arm as well as pooled, for the same reason the
# ranking is: a component that only writes ' Yes' damages the honest run on
# No-true items and *helps* it on Yes-true ones, and pooled those two average
# into something that looks like a mild consistent effect.
SURVIVORS = [r.component for r in all_hits][:10]

if SURVIVORS:
    print(f"{'component':<10} {'H->D recovery':>14} {'D->H damage':>12} "
          f"{'Yes-arm':>8} {'No-arm':>8}  consistent")
    for comp in SURVIVORS:
        fwd = attribution.median_recovery(
            [r for r in real_rows["D"] if r.component == comp and not r.is_null])
        damage = {}
        for it in SWEEP_ITEMS:
            entry = LD[it.item_id]
            cache_d = patcher.cache_slot(items.render(tok, it, "D"))
            ld_p = patcher.patched_logit_diff(
                items.render(tok, it, "H"), {comp: cache_d[comp]},
                entry["id_h"], entry["id_d"])
            # Fraction of the honest lead destroyed by importing the deceptive
            # activation. Same denominator as `recovery`, opposite direction.
            damage[it.item_id] = (entry["H"] - ld_p) / (entry["H"] - entry["D"])
        rev = float(np.median(list(damage.values())))
        arms = polarity.by_polarity(damage, [BY_ID[i] for i in damage])
        print(f"{comp.name:<10} {fwd:>14.3f} {rev:>12.3f} "
              f"{arms.mean_yes:>8.3f} {arms.mean_no:>8.3f}  "
              + ("yes" if rev > 0.1 and min(arms.mean_yes, arms.mean_no) > 0
                 else "NO -- one-run artefact" if rev <= 0.1
                 else "NO -- one arm only"))
else:
    print("no survivors to confirm -- nothing cleared 6's bars.")

no survivors to confirm -- nothing cleared 6's bars.


## 7. V6 — not evaluable on this model

**Status, 2026-09-04.** Under C2 gemma-3-4b-it answers `" No"` on 30/30 items (04's sweep, 04d
cell 7: C2 says Yes 0.00). It does not invert; it defaults to one token. A component set
found under C2 would be the set for "say No", and comparing it to the D set would not test
whether deception has its own mechanism. So the C2 sweep is **not run**, V6 is reported as
not evaluable, and the object found under D keeps PLAN2 §10's cautious name,
*output-override components*, until a working inversion control exists. The C1 persona
control below still runs.

The original section text is kept for the record:

## 7. V6 — does C2 separate from D?

**The gate that decides whether the word "deception" is allowed in the writeup** (PLAN2.md
§6). Not a comparison of curves — 04 already did that and found both crossing at L25 — but of
**component sets**.

Three steps, in this order:

1. **`k` is chosen by dose-response, not by taste.** The smallest set whose *joint* recovery
   reaches 0.5, capped at 10. Joint and not summed: ten heads that each "recover 0.1" are not
   a restoration, and adding them up is how a distributed non-result becomes a circuit.
2. **The set-level null**: 200 random sets of the same size, and the real set must clear the
   99th percentile. This is PLAN2.md §5's *"any k heads would do this"*.
3. **Jaccard(D-set, C2-set)**, against the overlap two random size-`k` sets get by chance.
   Two sets drawn from ~300 components share something occasionally; "they share three heads"
   means nothing without that baseline.

**If the sets are the same, that is the headline** and the object is renamed *output-override
components* throughout — not a footnote, and not a reason to go looking for a better persona
(§7.4).

`C1` closes the section as the persona control: same persona as D, instructed to answer
truthfully. Recovery structure there would mean the set is about wearing a persona, not about
the instruction to conceal.

In [40]:
# --- (1) the set, per condition, by dose-response -------------------------
#
# The selection rule is registered (exp2_spec.md 5.3): the smallest set whose
# joint recovery reaches SET_TARGET, taken in ranked order. That rule is NOT
# changed here -- the candidate ordering stays the pooled median, because
# re-ordering by arm status after the fact would be a different rule than the
# one registered. What is added is the report: how many of the selected set
# actually clear both arms, so a set carried by a polarity component is visible
# rather than inferred.
#
# Ranked order is NOT model order. The first run of this cell died at k=2 with
# nnsight's MissedProviderError: the D ranking begins L25M, L19M, and a trace
# that touches layer 25 before layer 19 asks for an output that has already
# gone by. `Patcher.patched_logit_diff` now writes a set in execution order
# (`attribution.trace_order`); the candidate list here stays in ranked order,
# because that is the order the registered rule adds components in.
SETS, CURVES_SET, JOINT = {}, {}, {}
for cond in CONDS:
    ranked = ranked_for(cond)
    ordered = [r.component for r in ranked]
    JOINT[cond] = attribution.joint_recovery_fn(patcher, SWEEP_ITEMS, LD, condition=cond)
    SETS[cond], CURVES_SET[cond] = attribution.select_set(ordered, JOINT[cond])
    armed = {r.component for r in attribution.hits(ranked)}
    n_armed = sum(c in armed for c in SETS[cond])
    print(f"{cond}: k={len(SETS[cond])}  joint recovery "
          f"{CURVES_SET[cond][-1]:.3f}  {[c.name for c in SETS[cond]]}")
    print(f"    {n_armed}/{len(SETS[cond])} of them clear both polarity arms"
          + ("" if n_armed == len(SETS[cond])
             else "  <- the rest recover in one polarity direction only"))
    if not armed:
        # 6 found no hits. The set above is then the top-k of a ranking with
        # nothing behind it, and the only thing this cell adds is the curve:
        # whether the top components restore LD jointly, or plateau, or cancel.
        print(f"    NO component cleared 6's bars under {cond}: this set is a "
              f"dose-response probe of the top-{len(SETS[cond])}, not a circuit. "
              f"Reached target {attribution.SET_TARGET:g}: "
              f"{'yes' if CURVES_SET[cond][-1] >= attribution.SET_TARGET else 'NO'}")

# The two conditions can stop at different k, so the shorter curve is padded
# with NaN rather than truncating the longer one -- a dose-response cut short
# at the other condition's k is a different plot than the one that was measured.
k_max = max(len(c) for c in CURVES_SET.values())
viz.series_line(
    list(range(1, k_max + 1)),
    {cond: CURVES_SET[cond] + [float("nan")] * (k_max - len(CURVES_SET[cond]))
     for cond in CONDS},
    title="dose-response: joint recovery as components are added",
    xaxis="set size k", yaxis="joint recovery",
).show()

# --- (2) the set-level null: would any k components do this? --------------
for cond in CONDS:
    k = len(SETS[cond])
    null_vals = [JOINT[cond](s) for s in
                 attribution.random_sets(COMPONENTS, k, n=20, seed=11)]
    bar = attribution.set_bar(null_vals, attribution.SET_NULL_PERCENTILE)
    real = CURVES_SET[cond][-1]
    print(f"{cond}: real {real:.3f} vs random-set p{attribution.SET_NULL_PERCENTILE:g} "
          f"{bar:.3f}  -> {'above' if real > bar else 'NOT ABOVE -- any k would do this'}")

# n=20, not the spec's 200: each random set costs n_items forward passes, and at
# 30 items 200 x 2 conditions is over an hour of GPU for a null on a number that
# is already decided by the 99th percentile of a small sample. If the real set
# is close to the bar, re-run this cell with n=200 before making any claim.
print("\n(null at n=20 for speed -- re-run at n=200 if the margin is narrow)")

D: k=2  joint recovery 0.722  ['L25M', 'L19M']
    0/2 of them clear both polarity arms  <- the rest recover in one polarity direction only
    NO component cleared 6's bars under D: this set is a dose-response probe of the top-2, not a circuit. Reached target 0.5: yes


D: real 0.722 vs random-set p99 0.336  -> above

(null at n=20 for speed -- re-run at n=200 if the margin is narrow)


In [41]:
# --- (3) V6: not evaluable -- see the markdown above ----------------------
# No C2 set exists to compare against. The two names 8 saves are kept so the
# payload schema does not change; both record "not evaluated".
overlap = None
chance = None
print("V6 not evaluable: the model says ' No' on 30/30 items under C2, so no C2 set was")
print("built. The D set is reported as output-override components (PLAN2 10).")
print(f"D set (k={len(SETS['D'])}) : {[c.name for c in SETS['D']]}")

# The persona control. Same persona as D, instructed truthfully -- so recovery
# structure here would mean the set is about wearing a persona, not about the
# instruction to conceal. Only the D set, only one pass: a control, not a search.
c1_rows = attribution.patch_sweep(
    patcher, SWEEP_ITEMS, LD, condition="C1", components=SETS["D"],
    family_of=FAMILY_OF, by_id=BY_ID,
    save_to=config.RESULTS / f"attribution_C1{_suffix}_{cfg.lens_id}.json",
    desc="C1 control")
print(f"\nC1 (persona-truthful) median recovery over the D set: "
      f"{attribution.median_recovery(c1_rows):.3f}")
print("Near zero is the expected result: under C1 there is nothing being suppressed, "
      "so there is nothing for these components to restore.")

V6 not evaluable: the model says ' No' on 30/30 items under C2, so no C2 set was
built. The D set is reported as output-override components (PLAN2 10).
D set (k=2) : ['L25M', 'L19M']


C1 control:   0%|          | 0/64 [00:00<?, ?it/s]


C1 (persona-truthful) median recovery over the D set: 0.366
Near zero is the expected result: under C1 there is nothing being suppressed, so there is nothing for these components to restore.


## 8. What is on disk, and what 06 gets

**Everything expensive was written as it was produced** — the null in §4, each condition's
sweep after every item in §5, the C1 control in §7. 04 lost its first V2 gate result to a
save cell at the bottom of the notebook: a save that only runs when nothing went wrong is
the opposite of what a save is for.

So this cell writes only the thing that needs the whole run — **the component set 06
ablates** — and re-reads it, because a file that cannot be loaded is not a saved result and
the cheapest moment to find that out is while the box is still up.

`just down` syncs `results/` off the box before destroying it.

In [42]:
import json

SET_JSON = config.RESULTS / f"attribution_sets{_suffix}_{cfg.lens_id}.json"

payload = {
    "model": cfg.name,
    # Which upstream run this set is about. 06 ablates whatever it finds here,
    # and an alleged set read as a v2 set is a claim about the wrong bank.
    "run_tag": RUN_TAG,
    "bank": BANK_JSON.name,
    "l_star_04": L_STAR,
    # The registered artefact, saved as registered (exp2_spec.md 2a) -- and the
    # polarity arms beside it, so 06 does not have to re-derive which subgroup
    # was primary and can see that the families were reported, not used.
    "families": FAM,
    "families_available": FAM_ALL,
    "item_ids": SEL.item_ids,
    "n_available": SEL.n_available,
    "subsample_seed": SEL.seed,
    "polarity_of": POLARITY_OF,
    "n_yes_true": sum(v == attribution.YES for v in POLARITY_OF.values()),
    "n_no_true": sum(v == attribution.NO for v in POLARITY_OF.values()),
    "whole_pairs": len(PAIR_INDEX),
    "bars": {"min_recovery": attribution.MIN_RECOVERY,
             "null_percentile": attribution.NULL_PERCENTILE,
             "set_null_percentile": attribution.SET_NULL_PERCENTILE,
             "set_target": attribution.SET_TARGET, "set_k_max": attribution.SET_K_MAX,
             "min_ld_gap": attribution.MIN_LD_GAP,
             "both_polarity_arms": True},
    "sets": {c: [comp.name for comp in SETS[c]] for c in CONDS},
    "layer_types": {c: [comp.layer_type for comp in SETS[c]] for c in CONDS},
    "dose_response": {c: CURVES_SET[c] for c in CONDS},
    "hits_D": [r.component.name for r in attribution.hits(RANKED[None])],
    # Both arms per hit. A set without its arm numbers is a list 06 cannot
    # re-check, and "it cleared both arms" is the claim that separates a belief
    # component from a ' Yes'-writer.
    "arms_D": {r.component.name: {"yes": r.median_yes, "no": r.median_no,
                                  "gap": r.arm_gap}
               for r in attribution.hits(RANKED[None])},
    "hits_D_pooled_only": [r.component.name
                           for r in attribution.hits(ranked_for("D", arms=False))],
    "jaccard_D_C2": overlap,                 # None: V6 not evaluable, C2 not swept
    "chance_overlap_p99": None if chance is None else float(np.percentile(chance, 99)),
    "v6_status": "not evaluable: model answers ' No' on 30/30 items under C2",
    "survivors_reverse_checked": [c.name for c in SURVIVORS],
}
SET_JSON.write_text(json.dumps(payload, indent=1))
print(f"sets + bars -> {SET_JSON}")

# Read it back. The bars travel with the sets on purpose: 06 must ablate a set
# that was selected under a stated rule, and a set without its rule is a list of
# names someone can re-derive differently.
check = json.loads(SET_JSON.read_text())
assert check["sets"]["D"] == [c.name for c in SETS["D"]]
print(f"reloaded: D set {check['sets']['D']}, bars {check['bars']}")
print(f"hits: {len(payload['hits_D'])} on all three bars, "
      f"{len(payload['hits_D_pooled_only'])} on the pooled median alone")
print(f"run: {RUN_TAG or 'v2'}, {len(SEL.item_ids)} of {SEL.n_available} items, "
      f"{payload['n_yes_true']} Yes-true / {payload['n_no_true']} No-true")

print(f"\neverything now in {config.RESULTS}:")
for f in sorted(config.RESULTS.glob("*")):
    print(f"  {f.name:<48} {f.stat().st_size / 1e3:>9.1f} kB")
print("\n`just down` syncs this directory off the box before destroying it.")

sets + bars -> /workspace/results/attribution_sets_alleged_gemma-3-4b-it.json
reloaded: D set ['L25M', 'L19M'], bars {'min_recovery': 0.2, 'null_percentile': 95.0, 'set_null_percentile': 99.0, 'set_target': 0.5, 'set_k_max': 10, 'min_ld_gap': 1.0, 'both_polarity_arms': True}
hits: 0 on all three bars, 0 on the pooled median alone
run: alleged, 32 of 141 items, 14 Yes-true / 18 No-true

everything now in /workspace/results:
  alleged_arm_gated_gemma-3-4b-it.json                 188.6 kB
  alleged_final_gate_gemma-3-4b-it.json                320.9 kB
  attribution_C1_alleged_gemma-3-4b-it.json             16.3 kB
  attribution_D_alleged_gemma-3-4b-it.json            2472.5 kB
  attribution_items_alleged_gemma-3-4b-it.json           3.3 kB
  attribution_null_alleged_gemma-3-4b-it.json         2472.6 kB
  attribution_sets_alleged_gemma-3-4b-it.json            7.8 kB
  belief_readout_all_alleged_gemma-3-4b-it.npz         741.7 kB
  belief_readout_all_gemma-3-4b-it.npz                 174.9 

## 9. Scratch

`patcher.cache_slot(prompt)` for every component at the slot, `patcher.patched_logit_diff`
for one run with any set replaced, `patcher.dla(prompt, id_h, id_d)` for the decomposition.

Worth trying, with the discipline attached:

- **Patch a different position.** `Patcher` reads position −1 by design (the answer slot is
  what makes `a_H` and `a_D` comparable). Whether a component matters one token earlier is a
  different experiment and belongs in a fresh notebook, not in a changed default here.
- **Sweep `MIN_RECOVERY`.** `rank(rows, min_recovery=…)` returns rows that carry the bar they
  were judged with, so a sensitivity curve cannot be mistaken for the primary result. The
  registered bar is 0.20 and stays 0.20 in every claim.
- **Ask why a component is a hit.** Its attention pattern at the slot, or what it writes in
  the unembed basis. That is 06 material and exploratory here.

Nothing below this line is depended on by anything above it.

In [43]:
# Scratch. One item, one component, both directions, printed in full.
_it = SWEEP_ITEMS[0]
_entry = LD[_it.item_id]
_comp = SETS["D"][0] if SETS.get("D") else attribution.Component(
    L_STAR, attribution.HEAD, 0, patcher.layer_types[L_STAR])

_cache_h = patcher.cache_slot(items.render(tok, _it, "H"))
_ld_p = patcher.patched_logit_diff(
    items.render(tok, _it, "D"), {_comp: _cache_h[_comp]}, _entry["id_h"], _entry["id_d"])
print(f"{_it.item_id}  patch {_comp.name} ({_comp.layer_type}) H->D")
print(f"  LD_H {_entry['H']:.3f}   LD_D {_entry['D']:.3f}   patched {_ld_p:.3f}")
print(f"  recovery {(_ld_p - _entry['D']) / (_entry['H'] - _entry['D']):.3f}")

AF011G_hgv_clutch  patch L25M (sliding_attention) H->D
  LD_H -4.000   LD_D -5.500   patched -7.125
  recovery -1.083
